In [ ]:
import numpy as np
import pandas as pd
import sys
import os
import pickle
import json
from collections import defaultdict
sys.path.append("../../experiment_util") 
import experiment_utils

In [ ]:
with open("../../../dir_paths.json") as f:
    config = json.load(f)
    dataframes_path = config["dataframes_path"]
    project_output_path = config["project_output_path"]

sampled_matched_perturbed_df = pd.read_pickle(dataframes_path + "/sampled_matched_perturbed_df.pkl")
post_embedding_df = pd.read_pickle(dataframes_path + "/text_embedding.pkl")
submission_embedding_df = pd.read_pickle(dataframes_path + "/submission_text_embedding.pkl")

text_embedding_df = pd.concat([post_embedding_df, submission_embedding_df], ignore_index=True)

In [ ]:
grouped = text_embedding_df.groupby("screen_name")["embedding"].apply(lambda x: np.mean(x.values))
screen_name_list = grouped.index.tolist()
user_embeds_list = grouped.values.tolist()

user_embedding_df = pd.DataFrame({
    "screen_name": screen_name_list,
    "user_embedding": user_embeds_list
})
user_embedding_df

In [ ]:
merged_df = pd.merge(sampled_matched_perturbed_df, user_embedding_df, left_on="user", right_on="screen_name")
merged_df = merged_df.dropna(subset=["user_embedding"])

In [ ]:
for dropped_percent in [0]:
    running_confusion_matrix_rf = np.zeros((2, 2), dtype=int)
    running_confusion_matrix_xgb = np.zeros((2, 2), dtype=int)

    f1_list_rf = []
    f1_list_xgb = []

    misclassified_users_rf_all_subs = defaultdict(int)
    misclassified_users_xgb_all_subs = defaultdict(int)
    for run in merged_df.run.unique():
        print(dropped_percent,"run:",run)
        seed = 100 + run

        df_russian_sampled = merged_df[(merged_df.run == run) & (merged_df.perturb_percent == dropped_percent) & (merged_df.russian == 1)].copy()
        df_sampled = merged_df[(merged_df.run == run) & (merged_df.perturb_percent == dropped_percent) & (merged_df.russian == 0)].copy()
        
        df_russian_sampled["feature_col"] = df_russian_sampled['user_embedding'].apply(lambda x: x.flatten())
        df_sampled["feature_col"] = df_sampled['user_embedding'].apply(lambda x: x.flatten())

        confusion_matrix_rf, confusion_matrix_xgb, overall_rf_f1, overall_xgb_f1, misclassified_users_rf, misclassified_users_xgb = experiment_utils.run_experiment(df_russian_sampled,df_sampled,seed = seed)

        f1_list_rf.append(overall_rf_f1)
        f1_list_xgb.append(overall_xgb_f1)
        running_confusion_matrix_rf += confusion_matrix_rf
        running_confusion_matrix_xgb += confusion_matrix_xgb
        for user in misclassified_users_rf:
            misclassified_users_rf_all_subs[user] += misclassified_users_rf[user]
        for user in misclassified_users_xgb:
            misclassified_users_xgb_all_subs[user] += misclassified_users_xgb[user]

    print("dropped_percent", dropped_percent)
    print("Final Confusion Matrix (Random Forest):\n", running_confusion_matrix_rf)    
    f1 = np.mean(f1_list_rf)
    print(f"Mean F1 Score (Random Forest): {f1:.2f}\n")

    print("Final Confusion Matrix (XGBoost):\n", running_confusion_matrix_xgb)        
    f1 = np.mean(f1_list_xgb)
    print(f"Mean F1 Score (XGBoost): {f1:.2f}\n")

    results = {
        "f1_list_rf":f1_list_rf,
        "f1_list_xgb":f1_list_xgb,
        "running_confusion_matrix_rf":running_confusion_matrix_rf,
        "running_confusion_matrix_xgb":running_confusion_matrix_xgb,
        "misclassified_users_rf_all_subs":misclassified_users_rf_all_subs,
        "misclassified_users_xgb_all_subs":misclassified_users_xgb_all_subs
    }

    output_dir = project_output_path + "/modernbert_embeds"
    os.makedirs(output_dir, exist_ok=True)
    with open(output_dir +'/' + 'full_results.pkl', 'wb') as f:
        pickle.dump(results, f)